# Winery E-commerce — Data Cleaning & Modeling

Starting from an anonymized dataset (no customer PII, no real product names/SKUs).

In [0]:
import pandas as pd

df_valid = pd.read_csv(
    '../data/processed/df_valid_anonymized.csv',
    low_memory=False
)

print(df_valid.shape)
df_valid.head()

In [0]:
df_valid.dtypes

## Date validation

In [0]:
# define which columns should be parsed back into real dates
date_cols = ['order_date', 'paid_date']

# parse each date column, turning any invalid/unparseable value into NaT instead of raising an error
for col_name in date_cols:
    df_valid[col_name] = pd.to_datetime(df_valid[col_name], errors='coerce')
    n_invalid = df_valid[col_name].isna().sum()
    print(f"{col_name}: {n_invalid} invalid/missing dates")

In [0]:
# check which order statuses correspond to the missing paid_date values
df_valid[df_valid['paid_date'].isna()]['status'].value_counts()

## Order status filtering

In [0]:
# check the full status distribution across all orders
df_valid["status"].value_counts()

In [0]:
n_before = len(df_valid)

# keep only orders with status "completed", since only these represent real, finalized revenue
df_valid = df_valid[df_valid["status"] == "completed"].copy()

n_excluded = n_before - len(df_valid)
print(f"{n_excluded} orders excluded (status != completed)")

## Structure & overview

In [0]:
# check the number of rows and columns in the cleaned dataset
df_valid.shape

In [0]:
# check column names and their data types
df_valid.dtypes

In [0]:
# look at a sample of rows
df_valid.head(10)

In [0]:
# look at a random sample of rows
df_valid.sample(5)

In [0]:
# get a concise summary: column names, non-null counts, and dtypes in one view
df_valid.info()

### Redundant columns

In [0]:
# check whether order_id and order_number are always identical
(df_valid['order_id'] == df_valid['order_number']).all()

In [0]:
# I drop one of the redundant columns
df_valid = df_valid.drop(columns=["order_number"])

In [0]:
# check if order_currency has more than one distinct value
df_valid['order_currency'].value_counts()

In [0]:
# Casting shipping_postcode to string
df_valid['shipping_postcode'] = df_valid['shipping_postcode'].astype('Int64').astype(str)

df_valid['shipping_postcode'].head()

In [0]:
# drop columns
df_valid = df_valid.drop(columns=[
    "tax_items",
    "fee_items",
    "coupon_items",
    "refund_items",
    "item_refunded",
    "item_refunded_qty",
    "transaction_id",
    "order_currency"
])

In [0]:
# convert order_id from float to integer first (removes the trailing .0), then to string
df_valid['order_id'] = df_valid['order_id'].astype('Int64').astype(str)

In [0]:
df_valid.info()

### Columns content

In [0]:
# loop through every column and show a quick summary of its content
for col_name in df_valid.columns:
    n_unique = df_valid[col_name].nunique()
    print(f"--- {col_name} ---")
    print(f"dtype: {df_valid[col_name].dtype}, unique values: {n_unique}")

    # if the column has few distinct values, show them all (useful for categorical columns)
    if n_unique <= 15:
        print(df_valid[col_name].value_counts(dropna=False))
    print()

In [0]:
# I correct the mistakes I notice:

# check if these three discount columns are always identical
print((df_valid['cart_discount'] == df_valid['order_discount']).all())
print((df_valid['order_discount'] == df_valid['discount_total']).all())
# output: True for both means I can safely drop 2 of the 3 columns

In [0]:
# check how many rows have a near-zero (but not exactly zero) item_subtotal
((df_valid['item_subtotal'] > 0) & (df_valid['item_subtotal'] < 0.01)).sum()
# output: rows here are effectively free items with a floating-point rounding artifact, not real prices

In [0]:
# I use is_gift_included (already derived from item_name in the private step) to confirm the pattern
df_valid[(df_valid['item_subtotal'] > 0) & (df_valid['item_subtotal'] < 0.01)]['is_gift_included'].value_counts()
# output: if most/all rows are True, confirms these are gift items, not pricing errors

In [0]:
# round near-zero item_subtotal/item_total values down to a true 0
# these are gift/promotional items with a technical near-zero price, not real charges
df_valid.loc[(df_valid['item_subtotal'] > 0) & (df_valid['item_subtotal'] < 0.01), 'item_subtotal'] = 0
df_valid.loc[(df_valid['item_total'] > 0) & (df_valid['item_total'] < 0.01), 'item_total'] = 0

In [0]:
# I prune the last useless columns
df_valid = df_valid.drop(columns=[
    "status",
    "shipping_tax_total",
    "cart_discount",
    "order_discount"
])

### Missing values

In [0]:
# FIX: no item_name column anymore — use item_product_id and category for context instead
df_valid[df_valid['item_quantity'].isna()][['order_id', 'item_product_id', 'category']]

In [0]:
# drop the rows that are empty line items (no product id, no quantity/totals)
df_valid = df_valid[df_valid['item_quantity'].notna()]

### Duplicates

In [0]:
# check for fully duplicated rows (identical values across every column)
df_valid.duplicated().sum()

In [0]:
# duplicates exist, so look at a sample of them to understand the pattern
df_valid[df_valid.duplicated(keep=False)].sort_values('order_id').head(10)

In [0]:
# check the pattern
df_valid[df_valid.duplicated(keep=False)][['order_id', 'category', 'product_type', 'item_quantity', 'item_total']].sort_values('order_id').head(20)

### Relational tables creation

#### orders table creation

In [0]:
# check if these "order-level" columns are truly identical across all line items of the same order
cols_to_check = ['shipping_total', 'fee_total', 'fee_tax_total', 'tax_total',
                  'discount_total', 'order_total', 'order_subtotal']

df_valid.groupby('order_id')[cols_to_check].nunique().max()
# result: if every column shows a max of 1, it confirms all these fields are identical within each order

In [0]:
orders = df_valid[[
    "order_id",
    "order_date",
    "paid_date",
    "customer_id",
    "shipping_total",
    "fee_total",
    "fee_tax_total",
    "tax_total",
    "discount_total",
    "order_total",
    "order_subtotal",
    "payment_method",
    "payment_method_title",
    "shipping_method",
    "shipping_postcode",
    "shipping_city",
    "shipping_state",
    "shipping_country",
    "order_type",
    "meta:_wc_order_attribution_device_type",
    "meta:_wc_order_attribution_referrer",
    "meta:_wc_order_attribution_session_count",
    "meta:_wc_order_attribution_session_pages",
    "meta:_wc_order_attribution_session_start_time",
    "meta:_wc_order_attribution_source_type",
    "meta:_wc_order_attribution_utm_source",
    "utm_source",
    "utm_medium",
    "utm_campaign"
]]

In [0]:
# drop duplicates in orders table (to have column order_id with unique values)
orders = orders.drop_duplicates(subset="order_id")

In [0]:
# check the result
len(orders) == orders["order_id"].nunique()

#### products table creation

In [0]:
# FIX: item_name no longer exists (real product names are private) — the products table
# is now built from the attributes we extracted from it, instead of the name itself
products = df_valid[[
    "item_product_id",
    "item_sku",
    "product_type",
    "format_size",
    "wine_type",
    "alcohol_level",
    "category",
    "is_subscription",
    "is_gift_included",
    "is_customizable"
]]

products = products.rename(columns={
    "item_product_id": "product_id",
    "item_sku": "product_sku"
})

products.info()

In [0]:
# drop duplicates to have unique primary key "product_id"
products = products.drop_duplicates(subset="product_id")

In [0]:
# check the result
len(products) == products["product_id"].nunique()

#### orders_products table creation

In [0]:
# create the table orders_products (bridge table: connects orders and products)
orders_products = df_valid[[
    "order_id",
    "item_product_id",
    "item_quantity",
    "item_subtotal",
    "item_subtotal_tax",
    "item_total",
    "item_total_tax"
]]

orders_products = orders_products.rename(columns={
    "item_product_id": "product_id",
    "item_quantity": "product_quantity",
    "item_subtotal": "product_subtotal",
    "item_subtotal_tax": "product_subtotal_tax",
    "item_total": "product_total",
    "item_total_tax": "product_total_tax"
})

orders_products.info()